# SAP News Intelligence — Embedding & Retrieval Pipeline

This notebook builds a retrieval-augmented pipeline over SAP-related news articles:

1. Load the scraped/cleaned articles
2. **Data cleaning** (dedupe, normalize text, drop broken rows, parse dates)
3. Chunk article text for embedding
4. Generate sentence embeddings (cached to disk so re-runs are fast)
5. Build a FAISS similarity index
6. Retrieve relevant chunks for a query
7. Run an LLM "opportunity agent" over the retrieved context

**Portability note:** paths are resolved relative to the project root (or a
`PROJECT_ROOT` environment variable) instead of being hardcoded to a specific
machine, so this notebook runs unmodified on any computer / grading environment.


## 1. Setup & configuration

In [7]:
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # reads a local .env file if present (API keys, overrides, etc.)

# Resolve the project root instead of hardcoding a machine-specific path.
# Override by setting PROJECT_ROOT in your environment / .env file.
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = DATA_DIR / "clean_data.json"
CHUNKED_DATA_PATH = DATA_DIR / "chunked_data.json"
EMBEDDINGS_PATH = DATA_DIR / "chunk_embeddings.npy"
FAISS_INDEX_PATH = DATA_DIR / "sap_intelligence.index"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
TOP_K_DEFAULT = 10
MIN_TEXT_LENGTH = 50  # filters out near-empty / broken scrapes

REQUIRED_COLUMNS = ["title", "source", "url", "published", "clean_text"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

Project root: /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook
Data dir:     /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook/data


## 2. Load raw data

Wrapped in a function with a clear error message instead of a bare
`pd.read_json` call, so a missing file fails fast and explains how to fix it.

In [8]:
def load_raw_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find input data at {path}. "
            "Set the PROJECT_ROOT environment variable, or place clean_data.json "
            "under ./data relative to this notebook."
        )
    df = pd.read_json(path)
    print(f"Loaded {len(df)} rows, {df.shape[1]} columns from {path.name}")
    return df


raw_df = load_raw_data(RAW_DATA_PATH)
raw_df.head()

Loaded 250 rows, 10 columns from clean_data.json


,title,description,content,source,published,url,text,clean_text,clean_length,data_source_type
0,From Intent to Impact: How Supply Chain Leader...,,Supply chain transformation today is a balanci...,SAP News,2026-02-02T13:15:00.000Z,https://news.sap.com/2026/02/how-supply-chain-...,From Intent to Impact: How Supply Chain Leader...,From Intent to Impact: How Supply Chain Leader...,6372,owned
1,"At Davos 2026, Sustainability Was Everywhere, ...",,Sustainability rarely took center stage at Dav...,SAP News,2026-02-04T12:15:00.000Z,https://news.sap.com/2026/02/davos-2026-sustai...,"At Davos 2026, Sustainability Was Everywhere, ...","At Davos 2026, Sustainability Was Everywhere, ...",8832,owned
2,SAP Introduces New Capabilities to Advance Pay...,,It’s no secret that organizations with fair an...,SAP News,2026-02-04T13:15:00.000Z,https://news.sap.com/2026/02/new-capabilities-...,SAP Introduces New Capabilities to Advance Pay...,SAP Introduces New Capabilities to Advance Pay...,5803,owned
3,SAP Is a Leader in the 2025 Gartner® Magic Qua...,,We are pleased to share that for the eighth co...,SAP News,2026-02-05T12:15:00.000Z,https://news.sap.com/2026/02/sap-a-leader-2025...,SAP Is a Leader in the 2025 Gartner® Magic Qua...,SAP Is a Leader in the 2025 Gartner® Magic Qua...,4028,owned
4,"AI, Sustainability, and the New Blueprint for ...",,"As we enter 2026, volatility and uncertainty h...",SAP News,2026-02-05T13:15:00.000Z,https://news.sap.com/2026/02/blueprint-for-sup...,"AI, Sustainability, and the New Blueprint for ...","AI, Sustainability, and the New Blueprint for ...",7653,owned


## 3. Data cleaning

The original notebook assumed the input was already "clean". In practice,
scraped news data usually has duplicate articles, missing fields, stray
HTML/whitespace, and inconsistent date formats — all of which hurt chunk
quality and retrieval relevance. This step:

- Drops rows missing essential fields (`clean_text`, `title`, `url`)
- Normalizes unicode characters and collapses whitespace/HTML remnants
- Drops articles that are suspiciously short (likely failed scrapes)
- De-duplicates by URL, and by `(title, source)` as a fallback
- Parses `published` into a real datetime (UTC) and flags rows that fail to parse
- Sorts chronologically and resets the index for clean, reproducible `chunk_id`s


In [9]:
def normalize_whitespace(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"<[^>]+>", " ", text)   # strip stray HTML tags
    text = re.sub(r"\s+", " ", text)       # collapse whitespace / newlines
    return text.strip()


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Expected columns missing from input data: {missing_cols}")

    before = len(df)

    # Drop rows missing essential fields
    df = df.dropna(subset=["clean_text", "title", "url"])

    # Normalize text fields
    df["title"] = df["title"].astype(str).map(normalize_whitespace)
    df["clean_text"] = df["clean_text"].astype(str).map(normalize_whitespace)
    df["source"] = df["source"].astype(str).str.strip()

    # Drop empty / too-short articles (likely failed scrapes)
    df = df[df["clean_text"].str.len() >= MIN_TEXT_LENGTH]

    # De-duplicate: same URL, or same (title, source) pair
    df = df.drop_duplicates(subset=["url"])
    df = df.drop_duplicates(subset=["title", "source"])

    # Standardize the published date; keep rows even if parsing fails, but flag them
    df["published"] = pd.to_datetime(df["published"], errors="coerce", utc=True)
    n_unparsed = int(df["published"].isna().sum())
    if n_unparsed:
        print(f"Warning: {n_unparsed} rows have an unparseable 'published' date.")

    df = df.sort_values("published", na_position="last").reset_index(drop=True)

    after = len(df)
    print(f"Cleaning removed {before - after} rows ({before} -> {after}).")
    return df


clean_df = clean_dataframe(raw_df)
clean_df.shape

Cleaning removed 0 rows (250 -> 250).


(250, 10)

## 4. Chunking

Same `RecursiveCharacterTextSplitter` settings as the original, now wrapped in functions and run on the *cleaned* dataframe.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def build_splitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


def chunk_dataframe(df: pd.DataFrame, splitter) -> pd.DataFrame:
    records = []
    for row_id, row in df.iterrows():
        text_chunks = splitter.split_text(row["clean_text"])
        for i, chunk in enumerate(text_chunks):
            records.append({
                "chunk_id": f"{row_id}_{i}",
                "text": chunk,
                "title": row["title"],
                "source": row["source"],
                "url": row["url"],
                "published": row["published"],
            })
    chunks_df = pd.DataFrame(records)
    chunks_df["chunk_length"] = chunks_df["text"].str.len()
    return chunks_df


splitter = build_splitter()
chunks_df = chunk_dataframe(clean_df, splitter)
print(chunks_df.shape)
chunks_df["chunk_length"].describe()

(1387, 7)


count    1387.000000
mean      916.317231
std       195.455952
min       137.000000
25%       991.000000
50%       995.000000
75%       998.000000
max      1000.000000
Name: chunk_length, dtype: float64

In [11]:
chunks_df.to_json(CHUNKED_DATA_PATH, orient="records", indent=4, date_format="iso")
print(f"Saved chunked data to {CHUNKED_DATA_PATH}")

Saved chunked data to /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook/data/chunked_data.json


## 5. Embeddings

Same embedding model as the original (`BAAI/bge-small-en-v1.5`), but:
- Wrapped in functions
- **Cached to disk** — re-running the notebook won't recompute embeddings for
  unchanged data, which matters once you have hundreds/thousands of chunks


In [12]:
from sentence_transformers import SentenceTransformer

def load_embedding_model(model_name=EMBEDDING_MODEL_NAME):
    return SentenceTransformer(model_name)


def compute_or_load_embeddings(texts, model, cache_path: Path = EMBEDDINGS_PATH) -> np.ndarray:
    if cache_path.exists():
        cached = np.load(cache_path)
        if cached.shape[0] == len(texts):
            print(f"Loaded cached embeddings from {cache_path} {cached.shape}.")
            return cached
        print("Cached embeddings size doesn't match current data; recomputing.")

    embeddings = model.encode(
        texts, show_progress_bar=True, normalize_embeddings=True
    ).astype("float32")
    np.save(cache_path, embeddings)
    print(f"Computed and cached embeddings {embeddings.shape} at {cache_path}.")
    return embeddings


embedding_model = load_embedding_model()
texts = chunks_df["text"].tolist()
embeddings = compute_or_load_embeddings(texts, embedding_model)
embeddings.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Computed and cached embeddings (1387, 384) at /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook/data/chunk_embeddings.npy.


(1387, 384)

## 6. FAISS vector index

The original notebook built this index twice (redundant cells); consolidated into one function here.

In [13]:
import faiss

def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    return index


faiss_index = build_faiss_index(embeddings)
faiss.write_index(faiss_index, str(FAISS_INDEX_PATH))
print(f"FAISS index built with {faiss_index.ntotal} vectors and saved to {FAISS_INDEX_PATH}")

FAISS index built with 1387 vectors and saved to /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook/data/sap_intelligence.index


## 7. Retrieval

Same behavior as the original `retrieve()`, plus similarity scores in the output and configurable de-duplication.

In [14]:
def retrieve(query: str, k: int = TOP_K_DEFAULT, dedupe_by_title: bool = True) -> pd.DataFrame:
    query_embedding = embedding_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = faiss_index.search(query_embedding, k)

    results = chunks_df.iloc[indices[0]].copy()
    results["score"] = scores[0]

    if dedupe_by_title:
        results = results.drop_duplicates(subset=["title"])

    return results[["title", "source", "published", "score", "text"]].reset_index(drop=True)


retrieve("What AI initiatives is SAP pursuing?")

,title,source,published,score,text
0,A Symphony of Partnership to Ring in the Era o...,SAP News,2026-05-13 16:00:00+00:00,0.814270,ecosystem‐led growth in support of SAP’s AI‐fi...
1,SAP Unveils Business AI Platform to Power the ...,SAP News,2026-05-13 16:01:00+00:00,0.807913,"AI into reality with SAP Business AI Platform,..."
2,The Future of the Enterprise Is Autonomous,SAP News,2026-05-13 10:00:00+00:00,0.799624,with SAP Signavio and SAP LeanIX further embed...
3,Meet the Hasso Plattner Founders’ Award Finali...,SAP News,2026-02-17 15:45:00+00:00,0.790351,"before the rise of large language models, SAP ..."
4,Moving Toward a More Autonomous Supply Chain,SAP News,2026-05-14 12:00:00+00:00,0.789407,"including SAP Cloud ERP Private, supporting SA..."
5,SAP at Hannover Messe 2026: Operationalizing A...,SAP News,2026-04-20 10:15:00+00:00,0.785960,"to resolve issues. With agentic AI, companies ..."


## 8. Opportunity agent (local LLM via Ollama)

The assignment requires an **open-source / locally-run model** — Anthropic, OpenAI,
and Gemini APIs are explicitly not allowed as the reasoning engine. This uses
**Ollama** running locally, talked to directly over its REST API (no extra
dependencies beyond `requests`, which is already in `pyproject.toml`).

Recommended models (set via `OLLAMA_MODEL` env var):
- `phi4-mini` — smallest, best default for CPU-only environments like Codespaces
- `qwen3:8b` / `llama3.1:8b` — better quality if you have more RAM/GPU

Before running this, make sure Ollama is installed, running, and has pulled a model:
```bash
ollama serve &
ollama pull phi4-mini
```


In [16]:
import requests
import time

OLLAMA_HOST    = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL   = os.getenv("OLLAMA_MODEL", "phi4-mini")
OLLAMA_TIMEOUT = int(os.getenv("OLLAMA_TIMEOUT", "300"))  # generous - CPU boxes are slow


def warmup_llm(model=OLLAMA_MODEL):
    """First call loads the model into memory - can take a minute+ on CPU.
    Run this once before any timed extraction so the real calls aren't slow."""
    print(f"Warming up {model}...")
    start = time.time()
    requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={"model": model, "prompt": "hi", "stream": False},
        timeout=OLLAMA_TIMEOUT,
    )
    print(f"Warmup done in {time.time() - start:.1f}s")


def ask_llm(prompt, model=OLLAMA_MODEL):
    resp = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=OLLAMA_TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()["response"]


warmup_llm()

Warming up phi4-mini...
Warmup done in 12.6s


In [17]:
def opportunity_agent(context: str) -> str:
    """Analyze retrieved context and surface business opportunities."""
    prompt = f"""You are a strategy analyst reviewing recent SAP-related news.

Identify and briefly justify:
1. Revenue opportunities
2. Market opportunities
3. Partnership opportunities
4. AI opportunities

Context:
{context}
"""
    return ask_llm(prompt)


# Example usage:
# results = retrieve("What AI initiatives is SAP pursuing?")
# context = "\n\n".join(results["text"].tolist())
# print(opportunity_agent(context))